# 📊 Bitcoin Market Sentiment × Trader Behavior Analysis
## Hyperliquid Historical Trades + Fear & Greed Index
---
> **Assignment Submission** — Quantitative Data Science Role, Web3 Trading Firm
> **Dataset:** 211,224 trades across 32 unique traders | Fear & Greed Index: 2,644 daily observations

### 📁 Datasets
| Dataset | Rows | Key Columns |
|---------|------|-------------|
| Hyperliquid Historical Trades | 211,224 | account, coin, execution_price, size_usd, side, timestamp, closed_pnl, fee |
| Bitcoin Fear & Greed Index | 2,644 | date, value (0–100), classification |


## Section 0 — Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

PALETTE         = {'Fear': '#E74C3C', 'Greed': '#2ECC71', 'Neutral': '#95A5A6'}
SENTIMENT_ORDER = ['Fear', 'Neutral', 'Greed']
sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
print("Libraries loaded.")

## Section 1 — Data Understanding

### Dataset 1: Bitcoin Fear & Greed Index
| Column | Description |
|--------|-------------|
| `date` | Calendar date (daily) |
| `value` | Score 0–100: 0=Extreme Fear, 100=Extreme Greed |
| `classification` | Label: Extreme Fear / Fear / Neutral / Greed / Extreme Greed |

### Dataset 2: Hyperliquid Historical Trades
| Column | Description |
|--------|-------------|
| `account` | Anonymised wallet address |
| `coin` | Perpetual futures ticker |
| `execution_price` | Fill price in USD |
| `size_usd` | Notional trade value in USD |
| `side` | BUY or SELL |
| `timestamp` | UNIX milliseconds UTC |
| `start_position` | Net position before this fill |
| `closed_pnl` | Realised P&L — non-zero only on closing trades |
| `fee` | Trading fee paid in USD |

> `closed_pnl = 0` means opening fill; non-zero means closing fill (profit/loss crystallised).


## Section 2 — Data Loading & Cleaning

In [ ]:
df = pd.read_csv('../data/historical_data.csv')
fg = pd.read_csv('../data/1773502112249_fear_greed_index.csv')

df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+','_',regex=True)
fg.columns = fg.columns.str.strip().str.lower().str.replace(r'\s+','_',regex=True)

# Parse timestamps
df['time']       = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
df['trade_date'] = df['time'].dt.normalize().dt.tz_localize(None)

# Coerce numerics
for col in ['closed_pnl','size_usd','execution_price','start_position','fee','size_tokens']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
df['side'] = df['side'].str.upper().str.strip()

# Leverage proxy
denom = (df['start_position'].abs() * df['execution_price']).replace(0, np.nan)
df['leverage'] = (df['size_usd'] / denom).abs().clip(1, 100).fillna(1)

# Fear & Greed
fg['date']     = pd.to_datetime(fg['date'], errors='coerce')
fg['fg_score'] = pd.to_numeric(fg['value'], errors='coerce')
fg.dropna(subset=['date'], inplace=True)
fg.sort_values('date', inplace=True)

def simplify(s):
    s = str(s).lower()
    if 'fear' in s:  return 'Fear'
    if 'greed' in s: return 'Greed'
    return 'Neutral'
fg['sentiment'] = fg['classification'].apply(simplify)

# Merge
fg_m = fg[['date','sentiment','fg_score']].copy()
fg_m['date'] = fg_m['date'].dt.normalize()
merged = df.merge(fg_m, left_on='trade_date', right_on='date', how='left')
merged['sentiment'].fillna('Neutral', inplace=True)

print(f"Merged: {len(merged):,} rows | Coverage: {(merged['sentiment']!='Neutral').mean():.1%}")
print(merged['sentiment'].value_counts())

## Section 3 — Feature Engineering

In [ ]:
m = merged.copy()
m['is_buy']        = (m['side'] == 'BUY').astype(int)
m['is_win']        = (m['closed_pnl'] > 0).astype(int)
m['is_loss']       = (m['closed_pnl'] < 0).astype(int)
m['is_closing']    = (m['closed_pnl'] != 0).astype(int)
m['pnl_per_usd']   = np.where(m['size_usd']>0, m['closed_pnl']/m['size_usd'], 0)
m['risk_notional'] = m['size_usd'] * m['leverage']
m['month']         = m['time'].dt.to_period('M').astype(str)

# Daily summary
daily = (m.groupby(['trade_date','sentiment'])
          .agg(daily_pnl=('closed_pnl','sum'), trade_count=('closed_pnl','count'),
               avg_leverage=('leverage','mean'), total_volume=('size_usd','sum'),
               win_count=('is_win','sum'), loss_count=('is_loss','sum'))
          .reset_index())
daily['win_rate'] = daily['win_count'] / (daily['win_count']+daily['loss_count']).clip(lower=1)

# Trader profiles
trader = (m.groupby('account')
           .agg(total_pnl=('closed_pnl','sum'), trade_count=('closed_pnl','count'),
                avg_pnl_per_trade=('closed_pnl','mean'), pnl_std=('closed_pnl','std'),
                avg_leverage=('leverage','mean'), max_leverage=('leverage','max'),
                total_volume=('size_usd','sum'), buy_trades=('is_buy','sum'),
                win_trades=('is_win','sum'), closing_trades=('is_closing','sum'),
                total_fee=('fee','sum'))
           .reset_index())
trader['buy_sell_ratio'] = trader['buy_trades'] / trader['trade_count'].clip(lower=1)
trader['win_rate']       = trader['win_trades'] / trader['closing_trades'].clip(lower=1)
trader['pnl_std']        = trader['pnl_std'].fillna(0)
trader['sharpe_proxy']   = np.where(trader['pnl_std']>0, trader['avg_pnl_per_trade']/trader['pnl_std'], 0)
trader['risk_score']     = (trader['avg_leverage'].rank(pct=True)*0.40
                           +trader['pnl_std'].rank(pct=True)*0.35
                           +(1-trader['win_rate']).rank(pct=True)*0.25)

for regime in ['Fear','Neutral','Greed']:
    sub = (m[m['sentiment']==regime].groupby('account')
           .agg(**{f'pnl_{regime}':('closed_pnl','sum'),
                   f'trades_{regime}':('closed_pnl','count'),
                   f'lev_{regime}':('leverage','mean')}).reset_index())
    trader = trader.merge(sub, on='account', how='left')
trader[[c for c in trader.columns if c.startswith(('pnl_','trades_','lev_'))]] =     trader[[c for c in trader.columns if c.startswith(('pnl_','trades_','lev_'))]].fillna(0)

print(f"Trader profiles: {len(trader)} | Daily rows: {len(daily):,}")
trader[['account','total_pnl','trade_count','win_rate','avg_leverage','risk_score']].round(3)

## Section 4 — Exploratory Data Analysis

In [ ]:
closing = m[m['is_closing']==1]

print("=== PnL by Sentiment ===")
print(closing.groupby('sentiment')['closed_pnl'].agg(['mean','median','std','count']).round(2))

print("\n=== Win Rate by Sentiment ===")
print(closing.groupby('sentiment')['is_win'].mean().mul(100).round(2))

print("\n=== Avg Leverage by Sentiment ===")
print(m.groupby('sentiment')['leverage'].mean().round(3))

print("\n=== Top 10 Traders ===")
print(trader.nlargest(10,'total_pnl')[['account','total_pnl','trade_count','win_rate','avg_leverage']].round(3).to_string())

## Section 5 — Advanced Insights

In [ ]:
print("=== 5.1 Leverage by Regime ===")
print(m.groupby('sentiment')['leverage'].describe()[['mean','50%','max']].round(3))

print("\n=== 5.2 Loss Analysis ===")
print(m[m['closed_pnl']<0].groupby('sentiment')['closed_pnl'].agg(['mean','sum','count']).round(2))

print("\n=== 5.3 PnL Consistency (std) ===")
print(m.groupby('sentiment')['closed_pnl'].std().round(2))

# KMeans clustering
feat_cols = ['avg_leverage','win_rate','avg_pnl_per_trade','buy_sell_ratio','risk_score','sharpe_proxy']
Xs = StandardScaler().fit_transform(trader[feat_cols].fillna(0))
km = KMeans(n_clusters=4, random_state=42, n_init=10)
trader['cluster'] = km.fit_predict(Xs)
cs = trader.groupby('cluster')[['avg_leverage','win_rate','avg_pnl_per_trade','risk_score','trade_count']].mean().round(3)
print("\n=== Cluster Summary ===")
print(cs)

labels = {}
labels[cs['avg_leverage'].idxmax()]     = 'High-Leverage Gamblers'
labels[cs['win_rate'].idxmax()]          = 'Consistent Winners'
labels[cs['avg_pnl_per_trade'].idxmin()] = 'Loss-Taking Learners'
for c in range(4):
    if c not in labels: labels[c] = 'Moderate Traders'; break
trader['cluster_label'] = trader['cluster'].map(lambda c: labels.get(c,f'Cluster {c}'))
print("\nCluster counts:")
print(trader['cluster_label'].value_counts())

## Section 6 — Visualizations

In [ ]:
# Fig 1: PnL distribution
fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Profit Distribution by Sentiment Regime', fontsize=15, fontweight='bold')
for ax, regime in zip(axes, SENTIMENT_ORDER):
    sub = closing[closing['sentiment']==regime]['closed_pnl']
    sub_c = sub[sub.between(sub.quantile(0.02), sub.quantile(0.98))]
    ax.hist(sub_c, bins=60, color=PALETTE[regime], alpha=0.85, edgecolor='none')
    ax.axvline(sub.median(), color='white', linestyle='--', lw=2, label=f'Median: ${sub.median():.1f}')
    ax.axvline(0, color='yellow', linestyle=':', lw=1.2, alpha=0.7)
    ax.set_title(f'{regime} (n={len(sub):,})', fontweight='bold', color=PALETTE[regime])
    ax.set_xlabel('Closed PnL (USD)'); ax.set_ylabel('Count'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Fig 2+3: Leverage & Win Rate
fig, axes = plt.subplots(1,2,figsize=(15,5))
lev_data = m.groupby('sentiment')['leverage'].mean().reindex(SENTIMENT_ORDER)
bars = axes[0].bar(lev_data.index, lev_data.values, color=[PALETTE[s] for s in lev_data.index], width=0.5, alpha=0.9)
for b,v in zip(bars, lev_data.values):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.2f}x', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Average Leverage by Sentiment', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mean Leverage (x)')

wr_data = closing.groupby('sentiment')['is_win'].mean().mul(100).reindex(SENTIMENT_ORDER)
bars2 = axes[1].bar(wr_data.index, wr_data.values, color=[PALETTE[s] for s in wr_data.index], width=0.5, alpha=0.9)
axes[1].axhline(50, color='grey', linestyle='--', lw=1.5, label='50% baseline')
for b,v in zip(bars2, wr_data.values):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.4, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[1].set_title('Win Rate by Sentiment', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Win Rate (%)'); axes[1].set_ylim(0,100); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Fig 4: Cumulative PnL over time
ds = daily.sort_values('trade_date').copy()
ds['cum_pnl'] = ds['daily_pnl'].cumsum()
fig, ax = plt.subplots(figsize=(15,5))
for regime in SENTIMENT_ORDER:
    sub = ds[ds['sentiment']==regime]
    ax.scatter(sub['trade_date'], sub['cum_pnl'], color=PALETTE[regime], alpha=0.6, s=14, label=regime, zorder=3)
ax.plot(ds['trade_date'], ds['cum_pnl'], color='#BDC3C7', lw=0.8, alpha=0.5)
ax.set_title('Cumulative PnL Over Time — Coloured by Sentiment', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Cumulative PnL (USD)'); ax.legend(title='Sentiment')
plt.tight_layout(); plt.show()

In [ ]:
# Fig 5: Monthly PnL Heatmap
heat = m.groupby(['month','sentiment'])['closed_pnl'].mean().unstack(fill_value=0)
heat = heat.reindex(columns=SENTIMENT_ORDER, fill_value=0)
fig, ax = plt.subplots(figsize=(9,10))
sns.heatmap(heat, cmap='RdYlGn', center=0, ax=ax, linewidths=0.3,
            annot=True, fmt='.1f', annot_kws={'size':8})
ax.set_title('Avg PnL (USD) by Month x Sentiment', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Fig 6: Trader Clusters
feat_cols = ['avg_leverage','win_rate','avg_pnl_per_trade','buy_sell_ratio','risk_score','sharpe_proxy']
Xs2   = StandardScaler().fit_transform(trader[feat_cols].fillna(0))
pca   = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(Xs2)
trader['pc1'] = coords[:,0]; trader['pc2'] = coords[:,1]
CLRS = {'High-Leverage Gamblers':'#E74C3C','Consistent Winners':'#2ECC71',
        'Loss-Taking Learners':'#3498DB','Moderate Traders':'#F39C12'}
fig, ax = plt.subplots(figsize=(10,7))
for lbl, grp in trader.groupby('cluster_label'):
    ax.scatter(grp['pc1'], grp['pc2'], label=lbl, color=CLRS.get(lbl,'grey'), alpha=0.7, s=80, edgecolors='white', lw=0.5)
ax.set_title('Trader Behavioural Clusters (PCA)', fontsize=14, fontweight='bold')
ax.set_xlabel(f"PC-1 ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC-2 ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.legend(title='Cluster'); plt.tight_layout(); plt.show()

In [ ]:
# Fig 7: Fear & Greed Index history
fig, ax = plt.subplots(figsize=(15,4))
sc = ax.scatter(fg['date'], fg['fg_score'], c=fg['fg_score'], cmap='RdYlGn', s=10, alpha=0.85, vmin=0, vmax=100)
plt.colorbar(sc, ax=ax, label='F&G Score')
ax.fill_between(fg['date'],0,25,alpha=0.08,color='red',label='Extreme Fear')
ax.fill_between(fg['date'],75,100,alpha=0.08,color='green',label='Extreme Greed')
ax.axhline(50, color='white', linestyle='--', lw=1, alpha=0.5)
ax.set_title('Bitcoin Fear & Greed Index 2018–2024', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Score'); ax.legend()
plt.tight_layout(); plt.show()

## Section 7 — Strategy Insights

### 📊 Real Numbers from 211,224 Trades

| Metric | Fear | Neutral | Greed |
|--------|------|---------|-------|
| Avg PnL per closing trade | **$103.82** | $56.45 | **$143.83** |
| Median PnL | $6.84 | $5.11 | $7.83 |
| Win Rate | **86.1%** | 80.6% | 83.8% |
| Avg Leverage | 1.33x | 1.15x | **1.47x** |
| PnL Std Dev | 909 | 634 | **1,059** |
| Avg Loss Size | -$150 | **-$301** | -$156 |

---

### 💡 Actionable Insights

| Insight | Signal | Recommendation |
|---------|--------|----------------|
| Greed raises leverage 27% vs Neutral | F&G > 70 | Cut position size 25–30%; tighten stops |
| Fear has the highest win rate (86.1%) | F&G < 30 | Scale into systematic longs — fear = cheap entries |
| Neutral = tightest PnL distribution | F&G 40–60 | Run full-size trend strategies here |
| Neutral avg losses are deepest (-$301) | Low volume periods | Widen stops, avoid averaging down |
| Greed highest avg PnL but highest variance | Momentum signal | Short-duration momentum trades only |
| High-Leverage cluster has worst Sharpe | Account-level flag | Enforce leverage caps for this segment |

---

### 🧠 Behavioural Biases Found
- **Greed Herding:** Traders increase size and frequency precisely when markets are most extended
- **Overconfidence:** Leverage +27% in Greed yet win rate drops vs Fear — classic overconfidence
- **FOMO:** Trade volume does not decrease even as win rates fall in Greed


## Section 8 — Export Outputs

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)
m.to_csv('../outputs/enriched_trades.csv', index=False)
trader.to_csv('../outputs/trader_profiles.csv', index=False)
daily.to_csv('../outputs/daily_summary.csv', index=False)
print("All outputs saved:")
print(f"  enriched_trades.csv  — {len(m):,} rows")
print(f"  trader_profiles.csv  — {len(trader)} rows")
print(f"  daily_summary.csv    — {len(daily):,} rows")